In [1]:

import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex":True,
    "font.family":"Helvetica",
    "font.size"  : 13,
})

# import equivariant steerable cnn
import escnn.gspaces
import escnn.nn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import from_numpy
from sklearn.model_selection import train_test_split

from tqdm import tqdm
# Checking if you have a GPU and CUDA set up -- if not, training will take longer by running on the CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.empty_cache()
# device = 'cpu'
# Set the plotting style
#plt.style.use('seaborn-poster')

from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from os import listdir
from os.path import isfile, join

from copy import deepcopy

#from getdist import MCSamples, plots

import pickle as pk

from hyperparameters import hparams

In [2]:
import sys
sys.path.append('./src')
from hyperparameters import hparams
from C3NN import C3NN
from load_data import load_data
from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn
import numpy as np
import random
from torch import from_numpy
import torch
import time, datetime
import cloudpickle as cpk

In [3]:
training_parameters, training_data = load_data(hparams)
training_parameters, training_data = training_parameters, training_data

In [4]:
hparams.order     = 4
# final val mse for 15 filters 4th order: 0.660266
hparams.num_filts = 25
class Regressor(nn.Module):
    def __init__(self, hidden_dim):
        super(Regressor, self).__init__()
        self.c3nn = C3NN(hparams)#.to(device='cuda')
        
        self.fc1 = nn.Linear(hparams.order*hparams.num_filts, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim, 8)

    def forward(self, x):
        x = self.c3nn(x)
        x = self.fc1(x)      # first linear layer
        x = self.relu(x)     # activation
        x = self.fc2(x)      # first linear layer
        x = self.relu(x)     # activation
        x = self.fc3(x)      # output layer
        return x

model = Regressor(64).to('cuda')

In [5]:
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
import math

def train_model(model, train_loader, val_loader, device,
                num_epochs=100,
                criterion=None,
                optimizer=None,
                lr=1e-3,
                # Early stopping:
                es_patience=10,
                es_min_delta=0.0,
                # LR scheduler on plateau:
                lr_factor=0.1,
                lr_patience=5,
                lr_min=1e-6,
                lr_threshold=1e-4):
    """
    Trains a regression model with MSE loss and reports MSE, RMSE, and MAE.
    Adds: Early stopping (on val MSE) and ReduceLROnPlateau scheduler.

    Returns:
        model (best weights restored), history (list of dicts per epoch)
    """
    model = model.to(device)
    if criterion is None:
        criterion = nn.MSELoss()
    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=lr_factor,
        patience=lr_patience,
        threshold=lr_threshold,
        threshold_mode='rel',
        min_lr=lr_min,
        verbose=False
    )

    print('Starting Training...')
    best_val = math.inf
    best_state = copy.deepcopy(model.state_dict())
    es_counter = 0
    history = []

    for epoch in range(num_epochs):
        # -------- TRAIN --------
        model.train()
        train_mse_sum, train_mae_sum, n_train = 0.0, 0.0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss = criterion(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                bsz = y.size(0)
                n_train += bsz
                train_mse_sum += torch.mean((pred - y) ** 2).item() * bsz
                train_mae_sum += torch.mean(torch.abs(pred - y)).item() * bsz

        train_mse = train_mse_sum / n_train
        train_rmse = train_mse ** 0.5
        train_mae = train_mae_sum / n_train

        # -------- VALIDATION --------
        model.eval()
        val_mse_sum, val_mae_sum, n_val = 0.0, 0.0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_mse_sum += torch.mean((pred - y) ** 2).item() * y.size(0)
                val_mae_sum += torch.mean(torch.abs(pred - y)).item() * y.size(0)
                n_val += y.size(0)

        val_mse = val_mse_sum / n_val
        val_rmse = val_mse ** 0.5
        val_mae = val_mae_sum / n_val

        # ---- Scheduler step on validation metric ----
        scheduler.step(val_mse)
        current_lr = optimizer.param_groups[0]['lr']

        # ---- Logging ----
        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train: MSE {train_mse:.6f} | RMSE {train_rmse:.6f} | MAE {train_mae:.6f} || "
            f"Val: MSE {val_mse:.6f} | RMSE {val_rmse:.6f} | MAE {val_mae:.6f} || "
            f"LR: {current_lr:.2e}"
        )

        history.append({
            "epoch": epoch + 1,
            "train_mse": train_mse, "train_rmse": train_rmse, "train_mae": train_mae,
            "val_mse": val_mse, "val_rmse": val_rmse, "val_mae": val_mae,
            "lr": current_lr
        })

        # ---- Early stopping (on val_mse) ----
        if val_mse < (best_val - es_min_delta):
            best_val = val_mse
            best_state = copy.deepcopy(model.state_dict())
            es_counter = 0
        else:
            es_counter += 1
            if es_counter >= es_patience:
                print(f"Early stopping triggered at epoch {epoch+1}. Best val MSE: {best_val:.6f}")
                break

    # restore best weights
    model.load_state_dict(best_state)
    return model, history


In [ ]:
# ----------------------------
# Convert tensors to Datasets & Loaders
# ----------------------------
training_parameters_mean = training_parameters.mean(0)
training_parameters_std = training_parameters.std(0)
training_parameters_stand = (training_parameters - training_parameters_mean) / training_parameters_std

X_train, X_val, y_train, y_val = train_test_split(training_data, training_parameters_stand, test_size=0.1, random_state=42)
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
device = 'cuda'
model, history = train_model(model, train_loader, val_loader, device, num_epochs=200, lr=1e-2)

Starting Training...
Epoch [1/200] Train: MSE 1.007025 | RMSE 1.003506 | MAE 0.860055 || Val: MSE 0.964384 | RMSE 0.982031 | MAE 0.837723 || LR: 1.00e-02
Epoch [2/200] Train: MSE 0.983510 | RMSE 0.991721 | MAE 0.849378 || Val: MSE 0.901673 | RMSE 0.949565 | MAE 0.807108 || LR: 1.00e-02
Epoch [3/200] Train: MSE 0.913472 | RMSE 0.955757 | MAE 0.809471 || Val: MSE 0.864998 | RMSE 0.930053 | MAE 0.781656 || LR: 1.00e-02
Epoch [4/200] Train: MSE 0.900519 | RMSE 0.948957 | MAE 0.800853 || Val: MSE 0.849342 | RMSE 0.921597 | MAE 0.773764 || LR: 1.00e-02
Epoch [5/200] Train: MSE 0.894348 | RMSE 0.945700 | MAE 0.795915 || Val: MSE 0.848804 | RMSE 0.921306 | MAE 0.766693 || LR: 1.00e-02
Epoch [6/200] Train: MSE 0.890780 | RMSE 0.943811 | MAE 0.792885 || Val: MSE 0.839649 | RMSE 0.916324 | MAE 0.765930 || LR: 1.00e-02
Epoch [7/200] Train: MSE 0.885565 | RMSE 0.941045 | MAE 0.790052 || Val: MSE 0.839462 | RMSE 0.916221 | MAE 0.762521 || LR: 1.00e-02
Epoch [8/200] Train: MSE 0.883464 | RMSE 0.93992

In [ ]:
%matplotlib inline
def plot_regression_predictions(y_true, y_pred, feature_names=None):
    """
    Plots prediction vs truth for each dimension of a multi-output regression.

    Args:
        y_true: torch.Tensor or np.ndarray of shape (N, D) with ground truth values
        y_pred: torch.Tensor or np.ndarray of shape (N, D) with predictions
        feature_names: list of strings for axis labels, length D (optional)
    """

    # Convert to numpy if torch tensors
    if hasattr(y_true, "detach"):
        y_true = y_true.detach().cpu().numpy()
    if hasattr(y_pred, "detach"):
        y_pred = y_pred.detach().cpu().numpy()

    D = y_true.shape[1]
    ncols = 4
    nrows = int(np.ceil(D / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
    axes = axes.flatten()

    for d in range(D):
        ax = axes[d]
        ax.scatter(y_true[:, d], y_pred[:, d], alpha=0.6)
        min_val = min(y_true[:, d].min(), y_pred[:, d].min())
        max_val = max(y_true[:, d].max(), y_pred[:, d].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--')  # y=x line
        label = feature_names[d] if feature_names else f"Dim {d+1}"
        ax.set_title(f"Prediction vs Truth: {label}")
        ax.set_xlabel("Truth")
        ax.set_ylabel("Prediction")

    # Remove unused subplots if any
    for d in range(D, len(axes)):
        fig.delaxes(axes[d])

    plt.tight_layout()
    plt.show()
    #plt.savefig('Plots/mse_testing.png')

In [ ]:
# Suppose y_true and y_pred are torch tensors of shape (N, 8)
model.eval()
y_pred = model(X_val)
plot_regression_predictions(y_val, y_pred,
                            feature_names=[f"Param {i+1}" for i in range(8)])

In [ ]:
y_val

In [ ]:
print(y_val-y_pred)

In [ ]:
# in the next step I could use the extracted summaries to train sbi now...